# Step 4 — Semantic Embeddings (LSA) & History-Profile Personalisation

**Purpose.** Step 3's TF-IDF features lifted cold-slate ranking by matching **exact** term IDs, but cold slates still score 0.10 vs 0.51 for warm ones. This step adds *semantic* matching: a 64-dimensional latent space built by truncated SVD (LSA) over the same TF-IDF name matrix, in which terms that co-occur across product names become comparable even when they never match exactly. No new dependencies, deterministic, and it reuses Step 3's machinery.

The latent space also unlocks a feature family TF-IDF cannot express: a **semantic user profile** — the mean embedding of the names of products the user interacted with — compared against the candidate's name embedding. That is personalisation which still works for the 80% of candidates that are cold: the user side comes from history, the candidate side from text alone.

| Feature | Definition |
|---|---|
| `q_sem_lsa_sim` | cosine between the query and the candidate name in the latent space |
| `u_sem_lsa_7d` | cosine between the candidate and the user's **last-7-days** history profile (all types recorded) |
| `u_sem_lsa_all` | cosine between the candidate and the user's **whole retained history** profile |

**Inputs:** Steps 1–3 outputs (`features.parquet` must already contain the Step 3 text columns) + `product_meta_data.txt`.
**Output:** `features.parquet` updated in place with the three new columns.
**Runtime:** ~30–45 min (most of it the sparse SVD); peak ~10 GB RAM.

In [1]:
import os, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import svds
import pyarrow as pa, pyarrow.compute as pc, pyarrow.csv as pacsv

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
OUT_DIR = DATA_DIR / 'processed'
SEED, K = 42, 64

cands = pd.read_parquet(OUT_DIR / 'candidates.parquet', columns=['user', 'wid', 'label'])
users = pd.read_parquet(OUT_DIR / 'users.parquet')
queries = pd.read_parquet(OUT_DIR / 'queries.parquet')
events = pd.read_parquet(OUT_DIR / 'events.parquet', columns=['user', 'wid', 'type', 'age_days'])
events = events[events['type'] < 3].reset_index(drop=True)
nU = len(users)
need_wids = np.union1d(cands['wid'].to_numpy(), events['wid'].to_numpy())   # candidates + history items
print(f'candidate rows {len(cands):,} | history events {len(events):,} | '
      f'products needing a name embedding {len(need_wids):,}')

Mounted at /content/drive
candidate rows 15,510,012 | history events 26,642,681 | products needing a name embedding 12,318,606


## 1. Parse product names (candidates **and** history items)

Same streamed parsing as Step 3, but over the union of candidate and history products — the user profiles need embeddings for history items too.

In [2]:
def tokens_of(col):
    parts = pc.split_pattern(pc.fill_null(col, '-1'), '\x18')
    flat = pc.list_flatten(parts)
    flat = pc.if_else(pc.equal(flat, ''), '-1', flat)
    return pc.list_value_length(parts).to_numpy().astype(np.int64), pc.cast(flat, pa.int64()).to_numpy()

t0 = time.time()
wid_parts, name_parts = [], []
reader = pacsv.open_csv(DATA_DIR / 'product_meta_data.txt',
    parse_options=pacsv.ParseOptions(delimiter='\t', quote_char=False),
    convert_options=pacsv.ConvertOptions(include_columns=['wid', 'name'],
        column_types={'wid': pa.int64(), 'name': pa.string()}))
off = 0
for batch in reader:
    tb = pa.Table.from_batches([batch])
    wid = tb['wid'].to_numpy()
    j = np.searchsorted(need_wids, wid)
    keep = (j < len(need_wids)) & (need_wids[np.minimum(j, len(need_wids) - 1)] == wid)
    if not keep.any():
        continue
    tb = tb.filter(pa.array(keep))
    ids = np.arange(off, off + len(tb), dtype=np.int64); off += len(tb)
    wid_parts.append(tb['wid'].to_numpy())
    ln, tk = tokens_of(tb.column('name'))
    name_parts.append((np.repeat(ids, ln), tk))

doc_wids = np.concatenate(wid_parts); del wid_parts
name_doc = np.concatenate([p[0] for p in name_parts])
name_tok = np.concatenate([p[1] for p in name_parts]); del name_parts
m = name_tok != -1
name_doc, name_tok = name_doc[m], name_tok[m]; del m; gc.collect()
nD = len(doc_wids)
print(f'parsed in {time.time() - t0:.0f}s: {nD:,} products with a name '
      f'({nD / len(need_wids):.1%} of needed) | name tokens {len(name_tok):,}')

parsed in 66s: 11,587,364 products with a name (94.1% of needed) | name tokens 323,410,436


## 2. TF-IDF matrix → truncated SVD → 64-d embeddings

The document-term matrix is built exactly as in Step 3 (binary TF, smoothed IDF, L2-normalised rows). `svds` factorises it as $A \approx U \Sigma V^\top$; item embeddings are the rows of $U\Sigma = AV$ and a query is projected with the same $V$, so both live in one space. Everything is then L2-normalised so dot products are cosines.

In [3]:
def dedupe(doc, code, V):
    key = np.unique(doc * V + code)
    return key // V, (key % V).astype(np.int64)

vocab = np.unique(name_tok)
d, c = dedupe(name_doc, np.searchsorted(vocab, name_tok), len(vocab))
del name_doc, name_tok; gc.collect()
df = np.bincount(c, minlength=len(vocab))
idf = (np.log((nD + 1) / (df + 1)) + 1).astype(np.float32)
norm = np.sqrt(np.bincount(d, weights=idf[c] ** 2, minlength=nD)); norm[norm == 0] = 1
A = sp.csr_matrix((idf[c] / norm[d], (d, c)), shape=(nD, len(vocab)), dtype=np.float32)
del d, c, norm; gc.collect()
print(f'A: {A.shape[0]:,} x {A.shape[1]:,} ({A.nnz:,} nnz)')

t0 = time.time()
v0 = np.random.default_rng(SEED).random(min(A.shape)).astype(np.float32)
_, S, Vt = svds(A, k=K, v0=v0, return_singular_vectors='vh')
Vt = Vt[np.argsort(-S)].astype(np.float32)
print(f'SVD in {time.time() - t0:.0f}s | top singular values {np.sort(S)[::-1][:5].round(1)}')

def l2rows(M):
    n = np.linalg.norm(M, axis=1, keepdims=True); n[n == 0] = 1
    return M / n

t0 = time.time()
item_emb = np.empty((nD, K), np.float32)                 # rows of A @ V.T, computed in chunks
for s in range(0, nD, 1_000_000):
    item_emb[s:s + 1_000_000] = A[s:s + 1_000_000] @ Vt.T
item_emb = l2rows(item_emb)
del A; gc.collect()
print(f'item embeddings in {time.time() - t0:.0f}s')

A: 11,587,364 x 963,741 (271,212,068 nnz)
SVD in 409s | top singular values [330.  234.2 189.3 167.9 159.9]
item embeddings in 21s


In [4]:
# query embeddings: same vocab/idf space, projected with the same Vt
qtext = users['qry'].map(queries.set_index('qry')['text'])
qsplit = qtext.str.split('\x18')
q_doc = np.repeat(np.arange(nU, dtype=np.int64), qsplit.str.len().to_numpy())
q_tok = pd.to_numeric(qsplit.explode(), errors='coerce').fillna(-1).to_numpy(np.int64)
q_doc, q_tok = q_doc[q_tok != -1], q_tok[q_tok != -1]
d, c_raw = dedupe(q_doc, q_tok, q_tok.max() + 1)
pos = np.searchsorted(vocab, c_raw)
in_v = (pos < len(vocab)) & (vocab[np.minimum(pos, len(vocab) - 1)] == c_raw)
d, pos = d[in_v], pos[in_v]
Q = sp.csr_matrix((idf[pos], (d, pos)), shape=(nU, len(vocab)), dtype=np.float32)
q_emb = l2rows(Q @ Vt.T)
del Q, q_doc, q_tok, d, c_raw, pos, in_v; gc.collect()
print('query embeddings done')

query embeddings done


## 3. Semantic user profiles from the history

A user's profile is the mean of the (normalised) name embeddings of the products in their history — one profile from the last 7 days (the window where *all* interaction types are recorded) and one from the whole retained history (order/cart-dominated further back, by the Step 1 retention windows). Repeated interactions with an item naturally weight it higher.

In [5]:
doc_of_ev = pd.Index(doc_wids).get_indexer(events['wid'])
ok_ev = doc_of_ev >= 0
eu, ed = events['user'].to_numpy()[ok_ev], doc_of_ev[ok_ev]
recent = (events['age_days'].to_numpy() < 7)[ok_ev]

def profile(mask):
    P = np.zeros((nU, K), np.float32)
    u, dd = eu[mask], ed[mask]
    for k in range(K):
        P[:, k] = np.bincount(u, weights=item_emb[dd, k], minlength=nU)
    return l2rows(P)

t0 = time.time()
prof_all, prof_7d = profile(np.ones(len(eu), bool)), profile(recent)
cov_all, cov_7d = (np.abs(prof_all).sum(1) > 0).mean(), (np.abs(prof_7d).sum(1) > 0).mean()
del events, doc_of_ev, ok_ev, eu, ed, recent; gc.collect()
print(f'profiles in {time.time() - t0:.0f}s | users with a profile: all-history {cov_all:.1%}, last 7d {cov_7d:.1%}')

profiles in 46s | users with a profile: all-history 99.9%, last 7d 88.2%


## 4. Score all pairs

In [6]:
doc_of = pd.Index(doc_wids).get_indexer(cands['wid'])
qi_all, ok = cands['user'].to_numpy(), doc_of >= 0
qi, di = qi_all[ok], doc_of[ok]

def pair_cos(U_mat):
    out = np.empty(len(qi), np.float32)
    for s in range(0, len(qi), 2_000_000):
        e = min(s + 2_000_000, len(qi))
        out[s:e] = np.einsum('ij,ij->i', U_mat[qi[s:e]], item_emb[di[s:e]])
    return out

t0 = time.time()
SEM_FEATURES = ['q_sem_lsa_sim', 'u_sem_lsa_7d', 'u_sem_lsa_all']
sem = {f: np.full(len(cands), np.nan, np.float32) for f in SEM_FEATURES}
sem['q_sem_lsa_sim'][ok] = pair_cos(q_emb)
sem['u_sem_lsa_7d'][ok] = pair_cos(prof_7d)
sem['u_sem_lsa_all'][ok] = pair_cos(prof_all)
del q_emb, prof_7d, prof_all, item_emb; gc.collect()
print(f'scored {len(qi):,} pairs in {time.time() - t0:.0f}s')
pd.DataFrame(sem).describe().T.round(4)

scored 15,439,674 pairs in 12s


,count,mean,std,min,25%,50%,75%,max
q_sem_lsa_sim,15439674.0,0.5387,0.2170,-0.3855,0.3948,0.5624,0.7072,1.0
u_sem_lsa_7d,15439674.0,0.3148,0.2551,-0.2782,0.1018,0.2690,0.4886,1.0
u_sem_lsa_all,15439674.0,0.3708,0.1958,-0.2338,0.2235,0.3458,0.4935,1.0


## 5. Signal check (validation slates, each feature alone)

In [7]:
def evaluate(df, score, k=10, seed=0):
    """Vectorised per-slate ranking metrics with random tie-breaking (identical to Steps 2–3)."""
    u = df['user'].to_numpy(); lab = df['label'].to_numpy().astype(np.float64)
    r = np.random.default_rng(seed).random(len(df))
    order = np.lexsort((r, -np.asarray(score, dtype=np.float64), u))
    u_s, lab_s = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(u_s[1:] != u_s[:-1]) + 1]
    rank = np.arange(len(u_s)) - np.repeat(start, np.diff(np.r_[start, len(u_s)]))
    top = rank < k
    disc = np.where(top, 1.0 / np.log2(rank + 2), 0.0)
    gid = np.repeat(np.arange(len(start)), np.diff(np.r_[start, len(u_s)]))
    dcg = np.bincount(gid, weights=(2 ** lab_s - 1) * disc)
    ideal_order = np.lexsort((-lab, u)); lab_i = lab[ideal_order]
    idcg = np.bincount(gid, weights=(2 ** lab_i - 1) * disc)
    has_pos = np.bincount(gid, weights=(lab_s > 0)) > 0
    has_buy = np.bincount(gid, weights=(lab_s == 3)) > 0
    hit = np.bincount(gid, weights=((lab_s > 0) & top)) > 0
    hit_buy = np.bincount(gid, weights=((lab_s == 3) & top)) > 0
    return {'NDCG@10': (dcg[has_pos] / idcg[has_pos]).mean(), 'HR@10': hit[has_pos].mean(),
            'Purchase-HR@10': hit_buy[has_buy].mean(), 'slates': int(has_pos.sum())}

split = pd.read_parquet(OUT_DIR / 'features.parquet', columns=['split'])['split'].to_numpy()
ev_df = pd.DataFrame({'user': cands['user'], 'label': cands['label']})
val_m = split == 'val'
single = pd.DataFrame({f: evaluate(ev_df[val_m], pd.Series(sem[f][val_m]).fillna(0)) for f in SEM_FEATURES}).T
single.loc['q_name_tfidf_sim (step 3)'] = evaluate(
    ev_df[val_m], pd.read_parquet(OUT_DIR / 'features.parquet', columns=['q_name_tfidf_sim'])['q_name_tfidf_sim'].to_numpy()[val_m])
single.loc['random'] = evaluate(ev_df[val_m], np.zeros(int(val_m.sum())))
single.sort_values('NDCG@10', ascending=False).round(4)

,NDCG@10,HR@10,Purchase-HR@10,slates
u_sem_lsa_7d,0.1157,0.2792,0.1931,25825.0
u_sem_lsa_all,0.1063,0.2677,0.1823,25825.0
q_name_tfidf_sim (step 3),0.0960,0.2444,0.1563,25825.0
q_sem_lsa_sim,0.0913,0.2377,0.1560,25825.0
random,0.0846,0.2326,0.1563,25825.0


## 6. Benchmark: Step-3 model vs + semantic features

Same training setup as Steps 2–3 (seed, split, shuffling, LambdaRank, early stopping).

In [8]:
feat = pd.read_parquet(OUT_DIR / 'features.parquet')
for f in SEM_FEATURES:
    feat[f] = sem[f]
del sem; gc.collect()
feat.to_parquet(OUT_DIR / 'features.parquet', index=False)
FEATURES_ALL = [c for c in feat.columns if c not in ('user', 'wid', 'label', 'split')]
FEATURES_BASE = [c for c in FEATURES_ALL if c not in SEM_FEATURES]
print(f'saved features.parquet with {len(FEATURES_ALL)} features '
      f'({len(FEATURES_BASE)} base+text + {len(SEM_FEATURES)} semantic)')

saved features.parquet with 49 features (46 base+text + 3 semantic)


In [9]:
import lightgbm as lgb, warnings
warnings.filterwarnings('ignore', message='.*eval_set.*deprecated')

def prepared(split_name):
    d = feat[feat['split'] == split_name]
    d = d.iloc[np.lexsort((np.random.default_rng(SEED).random(len(d)), d['user'].to_numpy()))]
    d = d[d.groupby('user')['label'].transform('max') > 0]
    return d, d.groupby('user', sort=False).size().to_numpy()

tr, g_tr = prepared('train'); va, g_va = prepared('val'); te, _ = prepared('test')
del feat; gc.collect()

def fit(cols, n_estimators=600):
    m = lgb.LGBMRanker(objective='lambdarank', n_estimators=n_estimators, learning_rate=0.05, num_leaves=63,
                       min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                       random_state=SEED, n_jobs=-1, verbose=-1)
    m.fit(tr[cols], tr['label'], group=g_tr, eval_set=[(va[cols], va['label'])], eval_group=[g_va],
          eval_at=[10], callbacks=[lgb.early_stopping(50, verbose=False)])
    return m

t0 = time.time(); base = fit(FEATURES_BASE); print(f'base (42+text): {time.time() - t0:.0f}s, iter {base.best_iteration_}')
t0 = time.time(); full = fit(FEATURES_ALL); print(f'+semantic: {time.time() - t0:.0f}s, iter {full.best_iteration_}')

results = pd.DataFrame({
    'random': evaluate(te, np.zeros(len(te))),
    'LightGBM base + text (46 features, step 3)': evaluate(te, base.predict(te[FEATURES_BASE])),
    'LightGBM + semantic (49 features)': evaluate(te, full.predict(te[FEATURES_ALL])),
}).T
results.round(4)

base (42+text): 247s, iter 386
+semantic: 262s, iter 412


,NDCG@10,HR@10,Purchase-HR@10,slates
random,0.0839,0.2268,0.1371,25822.0
"LightGBM base + text (46 features, step 3)",0.4144,0.7139,0.6695,25822.0
LightGBM + semantic (49 features),0.4154,0.7153,0.6675,25822.0


In [10]:
imp = pd.Series(full.booster_.feature_importance('gain'), index=FEATURES_ALL)
print('share of total split gain:')
(imp / imp.sum()).loc[SEM_FEATURES + ['q_name_tfidf_sim']].sort_values(ascending=False).round(4)

share of total split gain:


,0
q_name_tfidf_sim,0.0338
q_sem_lsa_sim,0.0045
u_sem_lsa_7d,0.0044
u_sem_lsa_all,0.0039


## 7. Cold-candidate segment

Same segmentation as Step 3: slates grouped by whether their positives are all cold (item in no user's history), mixed, or all warm.

In [11]:
hist_wids = np.unique(pd.read_parquet(OUT_DIR / 'events.parquet', columns=['wid'])['wid'].to_numpy())
j = np.searchsorted(hist_wids, te['wid'].to_numpy())
te_cold = (j >= len(hist_wids)) | (hist_wids[np.minimum(j, len(hist_wids) - 1)] != te['wid'].to_numpy())
pos = te['label'].to_numpy() > 0
cold_pos = pd.Series(np.where(pos, te_cold, np.nan), index=te.index)
seg_frac = cold_pos.groupby(te['user']).mean()
seg_of_user = pd.cut(seg_frac, [-0.01, 0.0, 0.999, 1.0], labels=['all warm', 'mixed', 'all cold'])
seg = te['user'].map(seg_of_user)

rows = {}
for s in ['all warm', 'mixed', 'all cold']:
    m = (seg == s).to_numpy()
    rows[f'base+text — positives {s}'] = evaluate(te[m], base.predict(te.loc[m, FEATURES_BASE]))
    rows[f'+semantic — positives {s}'] = evaluate(te[m], full.predict(te.loc[m, FEATURES_ALL]))
pd.DataFrame(rows).T.round(4)

,NDCG@10,HR@10,Purchase-HR@10,slates
base+text — positives all warm,0.5089,0.8119,0.7525,18466.0
+semantic — positives all warm,0.5090,0.8120,0.7477,18466.0
base+text — positives mixed,0.2664,0.7514,0.4479,3379.0
+semantic — positives mixed,0.2694,0.7535,0.4535,3379.0
base+text — positives all cold,0.1005,0.2271,0.1446,3977.0
+semantic — positives all cold,0.1046,0.2343,0.1529,3977.0


## 8. Takeaways

*(filled in after the full run)*